In [1]:
!pip install -q -U transformers datasets accelerate scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
file_path = "/content/drive/MyDrive/Abusive-Text-dravidianLangtech/train2.csv"

df = pd.read_csv(file_path)

df.columns = ["text", "label"]
df.head()


,text,label
0,இல்லை புரோ மலேசியாவில் டிரெண்ட் ஆயிடுச்சு அதில...,0
1,ரொம்ப சந்தோஷம் இதை பாக்க ஏன்டா தமிழ் பைத்தியங்...,0
2,அவளை மாறானவராக ஆகிறாள் மருத்துவ மனையில் சேர்த்...,1
3,நீ கட்டின புடவையோட வா நான் உன்னை பாத்துக்கறேன்...,0
4,இனி அடுத்தது சூர்யா தேவி நாலு பேர் கூட்டிட்டு ...,1


In [5]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)


In [6]:
model_name = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

In [7]:
def tokenize_function(texts):
    return tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=128
    )


In [8]:
train_dataset = Dataset.from_dict({
    "text": train_texts.tolist(),
    "label": train_labels.tolist()
})

val_dataset = Dataset.from_dict({
    "text": val_texts.tolist(),
    "label": val_labels.tolist()
})

train_dataset = train_dataset.map(lambda x: tokenize_function(x["text"]), batched=True)
val_dataset = val_dataset.map(lambda x: tokenize_function(x["text"]), batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


Map:   0%|          | 0/20756 [00:00<?, ? examples/s]

Map:   0%|          | 0/5190 [00:00<?, ? examples/s]

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)


pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds)
    }


In [11]:
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,

    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=100
)


In [12]:
data_collator = DataCollatorWithPadding(tokenizer)


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.418750,0.420509,0.821195,0.823708,0.789225,0.861343


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
trainer.evaluate()

{'eval_loss': 0.40793508291244507,
 'eval_accuracy': 0.8425818882466282,
 'eval_f1': 0.8450597382893988,
 'eval_precision': 0.8084179970972424,
 'eval_recall': 0.8851807707588399,
 'eval_runtime': 32.5301,
 'eval_samples_per_second': 159.545,
 'eval_steps_per_second': 9.991,
 'epoch': 3.0}

In [ ]:
trainer.save_model("/content/drive/MyDrive/muril_abuse_model")
tokenizer.save_pretrained("/content/drive/MyDrive/muril_abuse_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/muril_abuse_model/tokenizer_config.json',
 '/content/drive/MyDrive/muril_abuse_model/tokenizer.json')

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def predict_abusive(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()

    return "Abusive" if pred == 1 else "Non-Abusive"


In [ ]:
tests = [
    "நீ ஒரு முட்டாள்",
    "நல்ல நாள்",
    "உன் வேலை மோசம்",
    "வணக்கம் நண்பா",
    "போடா முட்டாள்",
    "இன்று மிகவும் சந்தோஷம்"
]

for t in tests:
    print(t, "->", predict_abusive(t))


நீ ஒரு முட்டாள் -> Abusive
நல்ல நாள் -> Non-Abusive
உன் வேலை மோசம் -> Abusive
வணக்கம் நண்பா -> Non-Abusive
போடா முட்டாள் -> Abusive
இன்று மிகவும் சந்தோஷம் -> Non-Abusive


In [ ]:
df.head(10)


,text,label
0,இல்லை புரோ மலேசியாவில் டிரெண்ட் ஆயிடுச்சு அதில...,0
1,ரொம்ப சந்தோஷம் இதை பாக்க ஏன்டா தமிழ் பைத்தியங்...,0
2,அவளை மாறானவராக ஆகிறாள் மருத்துவ மனையில் சேர்த்...,1
3,நீ கட்டின புடவையோட வா நான் உன்னை பாத்துக்கறேன்...,0
4,இனி அடுத்தது சூர்யா தேவி நாலு பேர் கூட்டிட்டு ...,1
5,எனக்கு மட்டும் தான் தப்பா கேக்குதா இத மட்டும் ...,0
6,நினைத்துப் பார்க்க முடியாதபடி அவர்கள் பணம் செல...,1
7,ஹேய் என்னமா பேசுற போன இன்டர்வியூ ல ஒன்னு பேசுற...,0
8,ஒருவேளை இந்த முட்டா கூதி இங்கேயும் கண்டெண்ட்கா...,1
9,தட்டி விடுங்க ஷார்மி அதுக்கு வேறு வேலை இல்லை அ...,1


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_path = "/content/drive/MyDrive/muril_abuse_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(197285, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [ ]:
print(predict_abusive("நீ ஒரு முட்டாள்"))   # expect 1
print(predict_abusive("நல்ல நாள்"))       # expect 0


Abusive
Non-Abusive


In [ ]:
import pandas as pd

test_df = pd.read_csv('/content/drive/MyDrive/Abusive-Text-dravidianLangtech/TestV2 - testV2.csv')

preds = [predict_abusive(t) for t in test_df["Text"]]

submission = pd.DataFrame({"label": preds})

submission.to_csv("KEC_LangTech.csv", index=False)
